# Biohub Exp109: Exp092 / Exp108 one-movie split

Fast last-slot candidate. This notebook does not rerun tracking. It merges two already-completed notebook outputs:

- Exp092 threshold + density-gap output for three datasets.
- Exp108 frozen-transition-aware output only for `6bba_05db0fb1`, where the frozen-frame intervention was largest.

The goal is to test whether the frozen-transition repair helps the large `6bba` movie without applying it to all outputs.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import pandas as pd

COLUMNS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
EXPECTED_DATASETS = ["44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"]

USE_EXP108_DATASETS = {"6bba_05db0fb1"}

WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUTPUT_PATH = WORKING_DIR / 'submission.csv'
RUN_STATS_PATH = WORKING_DIR / 'run_stats.csv'

EXP092_CANDIDATES = [
    Path('/kaggle/input/notebooks/dalloliogm/biohub-exp092-threshold-plus-density-gap-candidate/submission.csv'),
    Path('/kaggle/input/biohub-exp092-threshold-plus-density-gap-candidate/submission.csv'),
    Path('kaggle_outputs/biohub-exp092-threshold-plus-density-gap-candidate/submission.csv'),
]
EXP108_CANDIDATES = [
    Path('/kaggle/input/notebooks/dalloliogm/biohub-exp108-frozen-transition-aware/submission.csv'),
    Path('/kaggle/input/biohub-exp108-frozen-transition-aware/submission.csv'),
    Path('kaggle_outputs/biohub-exp108-frozen-transition-aware/submission.csv'),
]

def first_existing(paths: list[Path], label: str) -> Path:
    for path in paths:
        if path.exists():
            print(f'{label}: {path}')
            return path
    raise FileNotFoundError(f'Could not find {label}; tried: {[str(p) for p in paths]}')

EXP092_PATH = first_existing(EXP092_CANDIDATES, 'Exp092 submission')
EXP108_PATH = first_existing(EXP108_CANDIDATES, 'Exp108 submission')


In [ ]:
def load_submission(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if list(df.columns) != COLUMNS:
        raise ValueError(f'{path}: unexpected columns {list(df.columns)}')
    if df.isna().any().any():
        raise ValueError(f'{path}: contains missing values')
    datasets = sorted(df['dataset'].unique().tolist())
    if datasets != EXPECTED_DATASETS:
        raise ValueError(f'{path}: unexpected datasets {datasets}')
    return df

exp092 = load_submission(EXP092_PATH)
exp108 = load_submission(EXP108_PATH)

parts = []
source_by_dataset = {}
for dataset in EXPECTED_DATASETS:
    if dataset in USE_EXP108_DATASETS:
        src = 'exp108_frozen_transition_aware'
        part = exp108.loc[exp108['dataset'] == dataset].copy()
    else:
        src = 'exp092_threshold_density_gap'
        part = exp092.loc[exp092['dataset'] == dataset].copy()
    source_by_dataset[dataset] = src
    parts.append(part)

merged = pd.concat(parts, ignore_index=True)
merged['id'] = range(len(merged))
merged = merged[COLUMNS]

if sorted(merged['dataset'].unique().tolist()) != EXPECTED_DATASETS:
    raise AssertionError('dataset coverage changed during merge')
if merged['id'].iloc[0] != 0 or merged['id'].iloc[-1] != len(merged) - 1:
    raise AssertionError('row id renumbering failed')
if merged.duplicated(['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']).any():
    print('Warning: duplicate semantic rows exist; this can be normal for edge placeholder columns but should be inspected.')

summary = []
for dataset, g in merged.groupby('dataset', sort=True):
    nodes = int((g['row_type'] == 'node').sum())
    edges = int((g['row_type'] == 'edge').sum())
    divs = int(g.loc[g['row_type'] == 'edge'].groupby('source_id').size().ge(2).sum())
    summary.append({
        'dataset': dataset,
        'source': source_by_dataset[dataset],
        'rows': len(g),
        'nodes': nodes,
        'edges': edges,
        'division_like_sources': divs,
    })

stats = pd.DataFrame(summary)
print(stats.to_string(index=False))
print({'rows': int(len(merged)), 'nodes': int((merged.row_type == 'node').sum()), 'edges': int((merged.row_type == 'edge').sum())})


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_PATH, index=False)
stats.to_csv(RUN_STATS_PATH, index=False)

manifest = {
    'experiment': 'exp109_exp092_exp108_6bba05db_split',
    'exp092_path': str(EXP092_PATH),
    'exp108_path': str(EXP108_PATH),
    'use_exp108_datasets': sorted(USE_EXP108_DATASETS),
    'rows': int(len(merged)),
    'nodes': int((merged.row_type == 'node').sum()),
    'edges': int((merged.row_type == 'edge').sum()),
}
(WORKING_DIR / 'merge_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print(f'Wrote {OUTPUT_PATH}')
print(f'Wrote {RUN_STATS_PATH}')
print(json.dumps(manifest, indent=2))
display(pd.read_csv(OUTPUT_PATH, nrows=8))
